# 🔍 Auditoria de Meses Existentes por Ano — Camada Silver (SISCAN / 10ª CRS)

Este notebook **somente leitura** verifica quais meses estão presentes em cada ano das **tabelas mensais** da camada Silver (`tcc-analise-siscan-10crs.silver`) — aquelas em que a análise é feita por mês — destacando os **meses faltantes**, especialmente **Agosto/2022**, ausente na `bronze.exames_geral_raw`. Tabelas de análise apenas anual (`faixa_etaria`, `resultados_citopatologicos`) não dependem de meses e ficam fora da consulta. Nenhuma tabela é criada, alterada ou apagada: apenas consultas (`SELECT` / agregações).

### 🕵️ Auditoria de Integridade Temporal na Camada Bronze (Verificação de Agosto/2022)

**Objetivo:** Investigar diretamente no esquema da tabela Delta `bronze.exames_geral_raw` se a competência correspondente a **Agosto de 2022** esteve ausente desde a ingestão dos dados brutos originais extraídos do DATASUS/TABNET.

---

#### 🛠️ Etapas do Script de Verificação:
1. **Recuperação de Esquema:** Inspeciona todas as colunas dinâmicas carregadas na tabela `bronze.exames_geral_raw` via `spark.table().columns`.
2. **Filtragem por Ano:** Isola as colunas matriciais temporais correspondentes ao ano de **2022**.
3. **Teste Estrito de Presença:** Executa uma busca por expressão condicional (`any()`) procurando pelas colunas `AGOSTO/2022` ou `AGO/2022`.

---

#### 📌 Importância para o TCC e Governança de Dados:
* **Fidelidade à Fonte (*Single Source of Truth*):** Confirma se a série temporal histórica de 71 competências (de 72 meses esperados entre 2020 e 2025) decorre da ausência da coluna no arquivo físico original disponibilizado pelo DATASUS, e não de uma falha ou descarte inadvertido durante o processo de ETL/ELT[cite: 16, 18, 19].
* **Rastreabilidade Medallion:** Valida a integridade da **Camada Bronze** ao atestar que o pipeline não aplicou imputações arbitrárias ou "zeros falsos" antes da transposição (*unpivot*) na Camada Silver[cite: 1, 16, 18, 19].


In [0]:
# 1. Recuperar todas as colunas da tabela Bronze de exames gerais
colunas_bronze = spark.table(
    "`tcc-analise-siscan-10crs`.bronze.exames_geral_raw"
).columns

# 2. Filtrar apenas colunas que representam o ano de 2022
colunas_2022 = [col for col in colunas_bronze if "2022" in col]

print("=== Colunas encontradas para o ano de 2022 na Camada Bronze ===")
for col in sorted(colunas_2022):
  print(f" - {col}")

# 3. Testar explicitamente a presença de Agosto/2022
contem_agosto_2022 = any(
    "AGOSTO/2022" in col.upper() or "AGO/2022" in col.upper()
    for col in colunas_2022
)

if not contem_agosto_2022:
  print(
      "\n🚨 CONFIRMADO NA BRONZE: A coluna correspondente a AGOSTO/2022 NÃO"
      " EXISTE na tabela bronze.exames_geral_raw!"
  )
else:
  print("\n✅ Coluna de Agosto/2022 encontrada na Camada Bronze.")

## 1. Panorama Geral — Meses Presentes em Cada Ano

**Objetivo:** Para as tabelas Silver em que os **meses são essenciais** (análise mês a mês), agrupar por `ano` e listar os meses distintos presentes (`meses_presentes`), a quantidade (`qtd_meses`) e os meses do calendário que **não aparecem** (`meses_faltantes`) — evidenciando casos como **Agosto/2022**.

**Na consulta (tabelas mensais — meses obrigatórios):**
* `silver.exames_geral` — município × ano × mês.
* `silver.adequabilidade` — município × ano × mês.

**Fora da consulta (análise apenas anual):**
* `silver.faixa_etaria` e `silver.resultados_citopatologicos` possuem apenas `ano` (sem coluna `mes`), logo não são afetadas por lacunas mensais.
* Para 2025, espera-se um ano parcial — os "meses faltantes" refletem os dados disponíveis na fonte SISCAN/TABNET.

In [0]:
from pyspark.sql import functions as F

CATALOGO = "`tcc-analise-siscan-10crs`"

# Somente as tabelas Silver com granularidade MENSAL (ano + mês),
# onde a análise é realizada por mês e a ausência de um mês impacta o estudo.
# Tabelas de análise apenas anual (faixa_etaria, resultados_citopatologicos)
# são propositalmente EXCLUÍDAS desta consulta, pois não dependem de meses.
TABELAS_SILVER = {
    "silver.exames_geral": f"{CATALOGO}.silver.exames_geral",
    "silver.adequabilidade": f"{CATALOGO}.silver.adequabilidade",
}

# Calendário de referência: meses 1 a 12 de um ano completo
MESES_ANO = F.array(*[F.lit(m) for m in range(1, 13)])

resumos = []

for nome, tabela_fq in TABELAS_SILVER.items():
    df = spark.table(tabela_fq)

    # Tabelas com granularidade mensal: meses presentes e faltantes por ano
    resumo = (
            df.groupBy("ano")
            .agg(
                F.array_sort(F.collect_set("mes")).alias("meses_presentes"),
                F.count(F.lit(1)).alias("registros"),
            )
            .select(
                F.lit(nome).alias("tabela"),
                "ano",
                F.lit("ano + mês").alias("granularidade"),
                "meses_presentes",
                F.size("meses_presentes").alias("qtd_meses"),
                F.array_except(MESES_ANO, "meses_presentes").alias("meses_faltantes"),
                "registros",
            )
        )
    resumos.append(resumo)

# Consolida todas as tabelas em um único relatório
resumo_final = resumos[0]
for r in resumos[1:]:
    resumo_final = resumo_final.unionByName(r)

# Somente leitura: nenhuma gravação em tabela, apenas exibição
display(resumo_final.orderBy("tabela", "ano"))

## 2. Checagem Específica — Agosto de 2022 nas Tabelas Mensais

**Objetivo:** Confirmar se o mês de **Agosto de 2022** (que não existe na `bronze.exames_geral_raw`) também está ausente nas tabelas Silver com granularidade mensal (`exames_geral` e `adequabilidade`), detalhando os registros por mês de 2022.

In [0]:
# Tabelas Silver com granularidade mensal
TABELAS_MENSAIS = {
    "silver.exames_geral": f"{CATALOGO}.silver.exames_geral",
    "silver.adequabilidade": f"{CATALOGO}.silver.adequabilidade",
}

# 1. Detalhe dos meses de 2022 (registros por mês em cada tabela mensal)
detalhes = []
for nome, tabela_fq in TABELAS_MENSAIS.items():
    detalhes.append(
        spark.table(tabela_fq)
        .where(F.col("ano") == 2022)
        .groupBy("mes")
        .agg(F.count(F.lit(1)).alias("registros"))
        .select(F.lit(nome).alias("tabela"), "mes", "registros")
    )

detalhe_2022 = detalhes[0].unionByName(detalhes[1]).orderBy("tabela", "mes")
print("=== Registros por mês de 2022 nas tabelas mensais da Silver ===")
display(detalhe_2022)

# 2. Veredito: AGOSTO/2022 está presente?
print("\n=== Veredito — presença de AGOSTO/2022 na Silver ===")
for nome, tabela_fq in TABELAS_MENSAIS.items():
    tem_agosto_2022 = (
        spark.table(tabela_fq)
        .where((F.col("ano") == 2022) & (F.col("mes") == 8))
        .count()
        > 0
    )
    if tem_agosto_2022:
        print(f"✅ {nome}: AGOSTO/2022 presente.")
    else:
        print(
            f"🚨 {nome}: AGOSTO/2022 AUSENTE — a ausência da Bronze"
            " se propagou para a Silver."
        )

## 3. Auditoria na Camada Gold — Tabelas Mensais

**Objetivo:** Verificar, com o mesmo critério da Silver, se os **meses estão completos** nas tabelas Gold em que a análise é feita por mês — confirmando que a ausência de **Agosto/2022** também está presente na Gold (herdada da Bronze → Silver → Gold).

**Na consulta (tabelas mensais Gold):**
* `gold.fato_exames_temporais_territoriais` — série mensal completa (**espera os 12 meses** de cada ano, 2020–2025).
* `gold.fato_analise_outubro_rosa` — escopo da campanha Outubro Rosa (**espera apenas Outubro, Novembro e Dezembro** de cada ano).

**Fora da consulta (análise apenas anual):**
* `gold.fato_conformidade_faixa_etaria`, `gold.fato_indicador_populacao_alvo`, `gold.fato_adequabilidade_amostras` e `gold.fato_exames_alterados` possuem apenas `ano`, logo não são afetadas por lacunas mensais.

In [0]:
# Tabelas Gold com granularidade mensal e os meses esperados em cada ano:
# - fato_exames_temporais_territoriais: série mensal completa (meses 1 a 12).
# - fato_analise_outubro_rosa: escopo da campanha (apenas Outubro, Novembro e Dezembro).
TABELAS_GOLD_MENSAIS = {
    "gold.fato_exames_temporais_territoriais": (
        f"{CATALOGO}.gold.fato_exames_temporais_territoriais",
        [m for m in range(1, 13)],
    ),
    "gold.fato_analise_outubro_rosa": (
        f"{CATALOGO}.gold.fato_analise_outubro_rosa",
        [10, 11, 12],
    ),
}

resumos_gold = []
for nome, (tabela_fq, meses_esperados) in TABELAS_GOLD_MENSAIS.items():
    df = spark.table(tabela_fq)
    array_esperados = F.array(*[F.lit(m) for m in meses_esperados])
    resumo = (
        df.groupBy("ano")
        .agg(
            F.array_sort(F.collect_set("mes")).alias("meses_presentes"),
            F.count(F.lit(1)).alias("registros"),
        )
        .select(
            F.lit(nome).alias("tabela"),
            "ano",
            array_esperados.alias("meses_esperados"),
            "meses_presentes",
            F.size("meses_presentes").alias("qtd_meses"),
            F.array_except(array_esperados, "meses_presentes").alias("meses_faltantes"),
            "registros",
        )
    )
    resumos_gold.append(resumo)

resumo_gold = resumos_gold[0]
for r in resumos_gold[1:]:
    resumo_gold = resumo_gold.unionByName(r)

# Somente leitura: nenhuma gravação em tabela, apenas exibição
display(resumo_gold.orderBy("tabela", "ano"))

# Veredito: AGOSTO/2022 na tabela de série mensal da Gold
print("\n=== Veredito — presença de AGOSTO/2022 na Gold ===")
fq_temporais = TABELAS_GOLD_MENSAIS["gold.fato_exames_temporais_territoriais"][0]
tem_agosto_2022_gold = (
    spark.table(fq_temporais)
    .where((F.col("ano") == 2022) & (F.col("mes") == 8))
    .count()
    > 0
)
if tem_agosto_2022_gold:
    print("✅ gold.fato_exames_temporais_territoriais: AGOSTO/2022 presente.")
else:
    print(
        "🚨 gold.fato_exames_temporais_territoriais: AGOSTO/2022 AUSENTE — a lacuna"
        " da Bronze/Silver se propagou até a camada Gold."
    )
print(
    "ℹ️ gold.fato_analise_outubro_rosa: cobre apenas Out-Dez (campanha Outubro"
    " Rosa); Agosto não faz parte do escopo dessa tabela."
)